In [9]:
import numpy as np
tensor_1 = np.array([1,2,3])
tensor_2 = np.array([4,5,6])


print(f'Testing ipynb ... {np.dot(tensor_1,tensor_2)}')

Testing ipynb ... 32


In [11]:
from pathlib import Path
import rasterio
import numpy as np
import matplotlib.pyplot as plt

DATA_ROOT = Path("/mnt/d/Master/AN2/SEM1/CV3/DATA/Dataset_DACIA5")
DOCS_DIR = Path("../docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

sample_s2_path = DATA_ROOT / "Images_Sentinel2_12bands_GeoTIFF" / "Sentinel2_2020" / "Sentinel2_L2A_20200218.tif"
if not sample_s2_path.exists():
    sample_s2_path = sorted(list((DATA_ROOT / "Images_Sentinel2_12bands_GeoTIFF").rglob("*.tif")))[0]

sample_s1_path = DATA_ROOT / "Images_Sentinel1_2bands_GeoTIFF" / "Sentinel1_2020" / "Sentinel1_GRD_20200219.tif"
if not sample_s1_path.exists():
    sample_s1_path = sorted(list((DATA_ROOT / "Images_Sentinel1_2bands_GeoTIFF").rglob("*.tif")))[0]

with rasterio.open(sample_s2_path) as src_s2:
    s2_data = src_s2.read().astype(np.float32)

with rasterio.open(sample_s1_path) as src_s1:
    s1_data = src_s1.read().astype(np.float32)

def stretch(channel, lower=2.0, upper=98.0):
    vmin = np.percentile(channel, lower)
    vmax = np.percentile(channel, upper)
    if vmax == vmin:
        return np.zeros_like(channel)
    return np.clip((channel - vmin) / (vmax - vmin), 0.0, 1.0)

red = s2_data[3]
green = s2_data[2]
blue = s2_data[1]
nir = s2_data[7]

rgb = np.stack([stretch(red), stretch(green), stretch(blue)], axis=-1)
ndvi = np.clip((nir - red) / (nir + red + 1e-8), -0.2, 0.8)
ndvi_norm = (ndvi - (-0.2)) / (0.8 - (-0.2))

s1_vv = stretch(s1_data[0])
s1_vh = stretch(s1_data[1])

fig, axs = plt.subplots(2, 2, figsize=(4.8, 7.6))
plt.subplots_adjust(wspace=0.04, hspace=0.10, left=0.01, right=0.99, top=0.94, bottom=0.01)

axs[0, 0].imshow(rgb)
axs[0, 0].set_title("S2: RGB (B4-B3-B2)", fontsize=9, fontweight="bold", pad=4)
axs[0, 0].axis("off")

axs[0, 1].imshow(ndvi_norm, cmap="RdYlGn")
axs[0, 1].set_title("S2: Indice NDVI", fontsize=9, fontweight="bold", pad=4)
axs[0, 1].axis("off")

axs[1, 0].imshow(s1_vv, cmap="gray")
axs[1, 0].set_title("S1 SAR: Backscatter VV", fontsize=9, fontweight="bold", pad=4)
axs[1, 0].axis("off")

axs[1, 1].imshow(s1_vh, cmap="gray")
axs[1, 1].set_title("S1 SAR: Backscatter VH", fontsize=9, fontweight="bold", pad=4)
axs[1, 1].axis("off")

output_path = DOCS_DIR / "dacia5_inspection_compact.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight", pad_inches=0.02, transparent=True)
plt.close()